In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

### Loads the model state_dict

In [7]:
from unsloth import FastLanguageModel
from huggingface_hub import hf_hub_download
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

def load_adapter(huggingface_repo):
    # model,tokenizer=FastLanguageModel.from_pretrained(
    #     model_name="unsloth/gemma-3-270m-it",
    #     load_in_4bit=True,
    #     max_seq_length=2048,
    # )
    # model=FastLanguageModel.get_peft_model(
    #     model,
    #     r=16,
    #     target_modules=["q_proj","k_proj","v_proj",
    #                    "o_proj","gate_proj","down_proj","up_proj"],
    #     alpha_lora=32,
    #     lora_dropout=0,
    #     use_rslora=False,
    #     loftq_config=None,
    # )
    try:
        model_weights=hf_hub_download(
            repo_id=f"Srishtik/{huggingface_repo}",
            filename="adapter_model.safetensors"
        )
    except:
        model_weights=hf_hub_download(
            repo_id=f"Srishtik/{huggingface_repo}",
            filename="adapter_model.bin"
        )
    
    state_dict=load_file(model_weights)
    return state_dict
    

In [ ]:
# full_model=load_adapter("gemma-ag-news-finetuned_on_50K_samples")

### Extracts the lora deltas and checks if the layers count is 126

In [15]:
from safetensors.torch import load_file

def get_lora_deltas(repo_name: str, lora_alpha: int = 32, r: int = 16) -> dict:
    """Compute ΔW = lora_B @ lora_A * (alpha/r) directly from adapter weights."""
    from huggingface_hub import hf_hub_download

    path = hf_hub_download(
        repo_id  = f"Srishtik/{repo_name}",
        filename = "adapter_model.safetensors"
    )
    adapter_weights = load_file(path)
    scale = lora_alpha / r

    # Group A and B matrices by layer
    layers = {}
    for key, val in adapter_weights.items():
        if "lora_A" in key:
            base_key = key.replace("lora_A.default.weight", "").replace("lora_A.weight", "")
            layers.setdefault(base_key, {})["A"] = val.float()
        elif "lora_B" in key:
            base_key = key.replace("lora_B.default.weight", "").replace("lora_B.weight", "")
            layers.setdefault(base_key, {})["B"] = val.float()

    # Compute ΔW for each layer
    deltas = {}
    for base_key, mats in layers.items():
        if "A" in mats and "B" in mats:
            deltas[base_key] = scale * (mats["B"] @ mats["A"])  # (d_out, d_in)

    return deltas


# ── Usage ──
deltas_1 = get_lora_deltas("gemma3-270m-agnews-25k-1")
deltas_2 = get_lora_deltas("gemma3-270m-agnews-25k")

print(f"Adapter 1 layers: {len(deltas_1)}")  
print(f"Adapter 2 layers: {len(deltas_2)}")
print(f"Sample keys: {list(deltas_1.keys())[:3]}")

Adapter 1 layers: 126
Adapter 2 layers: 126
Sample keys: ['base_model.model.model.layers.0.mlp.down_proj.', 'base_model.model.model.layers.0.mlp.gate_proj.', 'base_model.model.model.layers.0.mlp.up_proj.']


### Appended the lora deltas into a list (deltas)

In [35]:
deltas=[]
deltas.append(deltas_1)
deltas.append(deltas_2)

In [16]:
def get_delta_weights(model):
    deltas={}
    for name,module in model.named_modules():
        if hasattr(module,"lora_A") and hasattr(module,"lora_B"):
            for layer_name in module.lora_A.keys():
                A=module.lora_A[layer_name].weight.detach().cpu().float()
                B=module.lora_B[layer_name].weight.detach().cpu().float()
                delta=B@A
                deltas[name]=delta
    return deltas
            

In [17]:
# model_1_deltas=get_delta_weights(model_1)
# model_2_deltas=get_delta_weights(model_2)

# 1) Merging Techniques

In [46]:
from copy import deepcopy
import torch.nn.functional as F
from typing import Dict, Optional

def apply_delta_to_base(base_state_dict,merged_deltas):
    merged=deepcopy(base_state_dict)
    for key in merged_deltas:
        if key in merged:
            merged[key]=(base_state_dict[key].float()+merged_deltas[key]).to(base_state_dict[key].dtype)
    return merged

## 1.1) Concatenation Merge

In [22]:
def concatenation_merge(lora_A1:Dict[str,torch.Tensor],lora_B1:Dict[str,torch.Tensor],lora_A2:Dict[str,torch.Tensor],lora_B2:Dict[str,torch.Tensor],alpha:float=1.0)->Dict[str,torch.Tensor]:
    merged_deltas={}
    keys=set(lora_A1.keys()) & set(lora_A2.keys())

    for key_A in keys:
        key_B=key_A.replace("lora_A","lora_B")
        if key_B not in lora_B1 or key_B not in lora_B1:
            continue
        A1=lora_A1[key_A].float()
        B1=lora_B1[key_B].float()
        A2=lora_A2[key_A].float()
        B2=lora_B2[key_B].float()

        A_cat=torch.cat([A1,A2],dim=0)
        B_cat=torch.cat([B1,B2],dim=1)

        delta=(alpha/(A1.shape[0]*2))*(B_cat@A_cat)
        base_key=key_A.replace("lora_A.","").replace("lora_A","")
        merged_deltas[base_key]=delta

    return merged_deltas

## 1.2) Linear Merge

### ΔW_merged = (1/n) * Σ ΔW_i

### Pros: Simple, fast

### Cons: Can cancel out task-specific directions


In [42]:
def linear_merge(delta1:Dict[str,torch.Tensor],delta2:Dict[str,torch.Tensor],weight1:float=0.5,weight2:float=0.5)->Dict[str,torch.Tensor]:
    merged={}
    keys=set(delta1.keys()) & set(delta2.keys())
    for key in keys:
        merged[key]=weight1*delta1[key].float()+weight2*delta2[key].float()

    return merged

## 1.3) SVD Merge

### - Project both deltas into shared singular subspace.

In [50]:
def svd_merge(delta1:Dict[str,torch.Tensor],delta2:Dict[str,torch.Tensor],weight1:float=0.5,weight2:float=0.5,rank:Optional[int]=None)->Dict[str,torch.Tensor]:
    merged={}
    keys=set(delta1.keys()) & set(delta2.keys())

    for key in keys:
        d1=delta1[key].float()
        d2=delta2[key].float()

        combined=weight1*d1+weight2*d2

        if combined.dim()<2:
            merged[key]=combined
            continue
        try:
            U,S,Vh=torch.linalg.svd(combined,full_matrices=False)
            r=rank if rank is not None else S.shape[0]
            r=min(r,S.shape[0])
            merged[key]=(U[:,:r]*S[:r].unsqueeze(0))@Vh[:r,:]
        except Exception:
            merged[key]=combined
    return merged

## 1.4) TIES Merge

### **Step 1** — TRIM: zero out lowest-magnitude parameters (keep top `density` fraction)

### **Step 2** — ELECT: majority vote on sign per parameter

### **Step 3** — MERGE: weighted mean of parameters that agree with elected sign

In [26]:
def ties_merge(deltas:list[Dict[str,torch.Tensor]],weights:Optional[list[float]]=None,density:float=0.2)->Dict[str,torch.Tensor]:
    n=len(deltas)
    if weights is None:
        weights=[1.0/n]*n
    all_keys=set(deltas[0].keys())
    for d in deltas[1:]:
        all_keys&=set(d.keys())
    merged={}
    for key in all_keys:
        tensors=[d[key].float() for d in deltas]
        
        ## TRIM
        trimmed=[]
        for t in tensors:
            flat=t.abs().flatten()
            if flat.numel()==0:
                trimmed.append(t)
                continue
            k=max(1,int(density*flat.numel()))
            threshold=torch.topk(flat,k).values.min()
            mask=t.abs()>=threshold
            trimmed.append(t*mask)
            
    ## ELECT 
        sign_sum=sum(torch.sign(t) for t in trimmed)
        elected_sign=torch.sign(sign_sum)
        elected_sign[elected_sign==0]=1.0
    
    ## MERGE
        numerator=torch.zeros_like(tensors[0]) # Sum of accepted weighted updates
        denominator=torch.zeros_like(tensors[0]) # Total weight of accepted adapters

        for w, t in zip(weights,trimmed):
            agree_mask=(torch.sign(t)==elected_sign).float()
            numerator+=w*t*agree_mask
            denominator+=w*agree_mask
        denominator=torch.clamp(denominator,min=1e-6)
        merged[key]=numerator/denominator
    
    return merged
    

## 1.5) DARE-MERGE

### Drop And REscale before merging

In [27]:
def dare_merge(
    deltas: list[Dict[str, torch.Tensor]],
    weights: Optional[list[float]] = None,
    density: float = 0.2,
    seed: int = 42,
) -> Dict[str, torch.Tensor]:
    n=len(deltas)
    if weights is None:
        weights=[1.0/n]*n
    all_keys=set(deltas[0].keys())
    for d in deltas[1:]:
        all_keys&=set(d.keys())

    merged={}
    rng=torch.Generator()
    rng.manual_seed(seed)

    for key in all_keys:
        tensors=[d[key].float() for d in deltas]
        result=torch.zeros_like(tensors[0])

        for w,t in zip(weights,tensors):
            mask=torch.bernoulli(
                torch.full(t.shape,density),generator=rng
            ).to(t.device)
            dare_delta=t*mask/(density+1e-8)
            result+=w*dare_delta
        merged[key]=result
    return merged

## 1.6) SLERP Merge

### SLERP interpolates along the great arc between two weight vectors,preserving the norm better than linear interpolation.

### t=0.0 → delta1, t=1.0 → delta2, t=0.5 → midpoint on sphere

### For each layer:
        omega = arccos(cosine_similarity(v1, v2))
        slerp = sin((1-t)*omega)/sin(omega) * v1 + sin(t*omega)/sin(omega) * v2
### Falls back to linear if vectors are nearly parallel (omega ≈ 0).

In [55]:
def slerp_merge(delta1:Dict[str,torch.Tensor],delta2:Dict[str,torch.Tensor],t:float=0.5,eps:float=1e-8)->Dict[str,torch.Tensor]:
    merged={}
    keys=set(delta1.keys())& set(delta2.keys())

    for key in keys:
        v1=delta1[key].float().flatten()
        v2=delta2[key].float().flatten()

        original_shape=delta1[key].shape

        n1=torch.norm(v1)
        n2=torch.norm(v2)

        if n1<eps or n2<eps:
            merged[key]=((1-t)*delta1[key].float()+t*delta2[key].float())
            continue
        v1_unit=v1/n1
        v2_unit=v2/n2

        dot=torch.clamp(torch.dot(v1_unit,v2_unit),-1.0+eps,1.0-eps)
        omega=torch.acos(dot)

        sin_omega=torch.sin(omega)

        if sin_omega.abs() < eps:
            merged[key]=((1-t)*delta1[key].float()+t*delta2[key].float())

        else:
            coeff1=torch.sin((1-t)*omega)/sin_omega
            coeff2=torch.sin(t*omega)/sin_omega

            interp_norm=(1-t)*n1+t*n2
            slerp_vec=(coeff1*v1_unit+coeff2*v2_unit)*interp_norm

            merged[key]=slerp_vec.reshape(original_shape)
            
    return merged

In [38]:
def merge_adapters(
    method: str,
    base_state_dict: Dict[str, torch.Tensor],
    deltas: list[Dict[str, torch.Tensor]],
    weights: Optional[list[float]] = None,
    **kwargs,
) -> Dict[str, torch.Tensor]:
    """
    Unified entry point for all merge methods.

    Args:
        method:                one of ['linear', 'svd', 'ties', 'dare', 'dare_ties', 'slerp']
        base_state_dict:       base model weights
        finetuned_state_dicts: list of finetuned model state dicts (2 for most methods)
        weights:               per-model weights (default: uniform)
        **kwargs:              method-specific args (density, rank, t, seed, etc.)

    Returns:
        merged state dict (ready to load into model)
    """
    n = len(deltas)
    if weights is None:
        weights = [1.0 / n] * n

    

    if method == "linear":
        assert n == 2, "Linear merge expects exactly 2 models"
        merged_delta = linear_merge(deltas[0], deltas[1], weights[0], weights[1])

    elif method == "svd":
        assert n == 2, "SVD merge expects exactly 2 models"
        merged_delta = svd_merge(
            deltas[0], deltas[1],
            weights[0], weights[1],
            rank=kwargs.get("rank", None)
        )

    elif method == "ties":
        merged_delta = ties_merge(deltas, weights=weights, density=kwargs.get("density", 0.2))

    elif method == "dare":
        merged_delta = dare_merge(
            deltas, weights=weights,
            density=kwargs.get("density", 0.2),
            seed=kwargs.get("seed", 42)
        )

    elif method == "dare_ties":
        merged_delta = dare_ties_merge(
            deltas, weights=weights,
            density=kwargs.get("density", 0.2),
            seed=kwargs.get("seed", 42)
        )

    elif method == "slerp":
        assert n == 2, "SLERP merge expects exactly 2 models"
        merged_delta = slerp_merge(deltas[0], deltas[1], t=kwargs.get("t", 0.5))

    else:
        raise ValueError(f"Unknown method: {method}. Choose from: linear, svd, ties, dare, dare_ties, slerp")

    return apply_delta_to_base(base_state_dict, merged_delta)

In [39]:
def upload_merged_model(
    merged_sd: dict,
    method: str,
    tokenizer,
    hf_token: str,
    base_repo: str = "unsloth/gemma-3-270m-it",
    your_hf_username: str = "Srishtik",
    max_seq_length: int = 2048,
    dtype=torch.float16,
    push_to_hub: bool = True,
    save_local: bool = False,
    local_dir: str = "./merged_models",
):
    """
    Loads merged state dict into a fresh base model and uploads to HuggingFace.

    Args:
        merged_sd           : output of merge_adapters()
        method              : merge method name — used for repo naming
        tokenizer           : tokenizer from your training run
        hf_token            : your HuggingFace write token
        base_repo           : base model to load architecture from
        your_hf_username    : your HF username
        max_seq_length      : must match training config
        dtype               : float16 recommended for upload
        push_to_hub         : whether to push to HF Hub
        save_local          : whether to also save locally
        local_dir           : parent dir for local saves
    """
    import os
    from unsloth import FastLanguageModel

    repo_name = f"{your_hf_username}/gemma-3-270m-{method}-merged"
    print(f"[upload] Preparing model for method='{method}' → {repo_name}")

    # ── Load fresh base model to receive merged weights ──
    model, _ = FastLanguageModel.from_pretrained(
        model_name     = base_repo,
        max_seq_length = max_seq_length,
        load_in_4bit   = False,
        dtype          = dtype,
    )

    # ── Cast merged_sd to match model dtype before loading ──
    target_dtype = next(model.parameters()).dtype
    cast_sd = {
        k: v.to(target_dtype) if v.is_floating_point() else v
        for k, v in merged_sd.items()
    }

    # ── Load merged weights ──
    missing, unexpected = model.load_state_dict(cast_sd, strict=False)
    if missing:
        print(f"  [warn] Missing keys  : {len(missing)}  (e.g. {missing[:3]})")
    if unexpected:
        print(f"  [warn] Unexpected keys: {len(unexpected)} (e.g. {unexpected[:3]})")

    model.eval()

    # ── Save locally ──
    if save_local:
        save_path = os.path.join(local_dir, f"gemma-3-270m-{method}-merged")
        os.makedirs(save_path, exist_ok=True)
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  [local] Saved to {save_path}")

    # ── Push to HuggingFace Hub ──
    if push_to_hub:
        model.push_to_hub(repo_name, token=hf_token, private=False)
        tokenizer.push_to_hub(repo_name, token=hf_token, private=False)
        print(f"  [hub] Pushed → https://huggingface.co/{repo_name}")

    # ── Free memory ──
    del model, cast_sd
    torch.cuda.empty_cache()

    return repo_name

In [48]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/gemma-3-270m-it")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [33]:
from transformers import AutoModelForCausalLM
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/gemma-3-270m-it",
    torch_dtype=torch.float16,
    device_map="cpu",
)
base_sd = {k: v.cpu() for k, v in base_model.state_dict().items()}
del base_model
torch.cuda.empty_cache()

print(f"Base keys: {len(base_sd)}")
print(f"Sample base keys: {list(base_sd.keys())[:3]}")

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Base keys: 237
Sample base keys: ['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.k_proj.weight']


In [40]:
def normalize_key(key: str) -> str:
    key = key.replace("base_model.model.", "")
    key = key.rstrip(".")
    return key + ".weight"

deltas_1_norm = {normalize_key(k): v for k, v in deltas_1.items()}
deltas_2_norm = {normalize_key(k): v for k, v in deltas_2.items()}


In [51]:
HF_TOKEN = None ## Insert your own token
methods  = ["linear", "svd", "ties", "dare", "slerp"]

for method in methods:
    
    merged_sd = merge_adapters(
        method = method,
        base_state_dict = base_sd,
        deltas = [deltas_1_norm, deltas_2_norm],
        weights = [0.5, 0.5],
        density  = 0.2,   # ties / dare / dare_ties
        rank = 16,    # svd
        t  = 0.5,   # slerp
        seed = 42,    # dare
    )

    upload_merged_model(
        merged_sd = merged_sd,
        method    = method,
        tokenizer = tokenizer,
        hf_token  = HF_TOKEN,
    )

[upload] Preparing model for method='linear' → Srishtik/gemma-3-270m-linear-merged
==((====))==  Unsloth 2026.6.1: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Saved model to https://huggingface.co/Srishtik/gemma-3-270m-dare-merged


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  [hub] Pushed → https://huggingface.co/Srishtik/gemma-3-270m-dare-merged


TypeError: 'builtin_function_or_method' object is not iterable

In [56]:
HF_TOKEN = None ## Insert your own token
methods  = ["slerp"]

for method in methods:
    
    merged_sd = merge_adapters(
        method = method,
        base_state_dict = base_sd,
        deltas = [deltas_1_norm, deltas_2_norm],
        weights = [0.5, 0.5],
        density  = 0.2,   # ties / dare / dare_ties
        rank = 16,    # svd
        t  = 0.5,   # slerp
        seed = 42,    # dare
    )

    upload_merged_model(
        merged_sd = merged_sd,
        method    = method,
        tokenizer = tokenizer,
        hf_token  = HF_TOKEN,
    )

[upload] Preparing model for method='slerp' → Srishtik/gemma-3-270m-slerp-merged
==((====))==  Unsloth 2026.6.1: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/548 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Srishtik/gemma-3-270m-slerp-merged


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  [hub] Pushed → https://huggingface.co/Srishtik/gemma-3-270m-slerp-merged


## 